# 00 - Phase 0: data foundation

**Before running**: *Add Input* -> a Kaggle Dataset containing `training_set_15000.csv` and `unseen_set_1000.csv`.
Internet and GPU are not needed.

Produces (in `/kaggle/working/processed/`): `train_processed.csv` (group ids, medoid flag, split, parsed citations),
`test_processed.csv` (overlap with train flagged), `split.json` (IDs of train/dev/test - **keep this, the report must reproduce it exactly**),
`phase0_report.json`. The split is by *question group*, seed 42, so no question text appears in both train and test.

In [1]:
!pip install -q rouge-score sacrebleu

In [2]:
%%writefile sources.py
"""Parse the `final_answer_sources` column of the training CSV.

The column is a stringified list of dict-strings, newline separated (no commas), e.g.
    ["{'source': 'https://workdrive.zoho.in/file/<id>', 'page': '1,2', 'sourceName': '...'}"
     "{'source': ..., 'page': None, ...}"]
Some entries are junk (free-text notes, zoho sheets, leading spaces, empty strings).
"""
import ast
import re

BLOCK_RE = re.compile(r"\{[^{}]*\}")
FIELD_RE = {
    k: re.compile(r"'%s':\s*(?:'((?:[^'\\]|\\.)*)'|\"((?:[^\"\\]|\\.)*)\"|(None|\d+))" % k)
    for k in ("source", "page", "sourceName", "sourceType")
}
FILE_ID_RE = re.compile(r"/file/([A-Za-z0-9]+)")


def _fields_from_block(block):
    try:
        d = ast.literal_eval(block)
        if isinstance(d, dict):
            return d
    except Exception:
        pass
    d = {}
    for k, rx in FIELD_RE.items():
        m = rx.search(block)
        if m:
            v = m.group(1) if m.group(1) is not None else (m.group(2) if m.group(2) is not None else m.group(3))
            d[k] = None if v == "None" else v
    return d


def parse_pages(page):
    """'45, 46' -> [45, 46]; '3-5' -> [3, 4, 5]; None -> []."""
    if page is None:
        return []
    s = str(page)
    pages = []
    for part in re.split(r"[,;&]|\band\b", s):
        part = part.strip()
        m = re.fullmatch(r"(\d+)\s*[-–]\s*(\d+)", part)
        if m and int(m.group(2)) - int(m.group(1)) < 200:
            pages.extend(range(int(m.group(1)), int(m.group(2)) + 1))
        elif re.fullmatch(r"\d+", part):
            pages.append(int(part))
    return sorted(set(pages))


def classify(url):
    url = (url or "").strip()
    if not url.lower().startswith("http"):
        return "invalid", None
    if re.match(r"https?://workdrive\.zoho(external)?\.in/", url):
        m = FILE_ID_RE.search(url)
        return ("zoho", m.group(1)) if m else ("invalid", None)
    if "sheet.zoho" in url or "docs.zoho" in url:
        return "zoho_other", None
    return "external", None


def parse_sources(cell):
    """Return a list of citation dicts: kind, file_id, url, pages, page_raw, name, source_type."""
    out = []
    for block in BLOCK_RE.findall(cell or ""):
        d = _fields_from_block(block)
        raw = (d.get("source") or "").strip()
        # some entries concatenate several URLs in one field
        urls = [u.strip() for u in re.split(r"(?=https?://)", raw) if u.strip()] or [raw]
        for url in urls:
            kind, fid = classify(url)
            out.append({
                "kind": kind,
                "file_id": fid,
                "url": url,
                "pages": parse_pages(d.get("page")),
                "page_raw": None if d.get("page") is None else str(d.get("page")),
                "name": (d.get("sourceName") or "").strip(),
                "source_type": d.get("sourceType"),
            })
    return out

Writing sources.py


In [3]:
%%writefile phase0_prepare.py
"""Phase 0: data foundation (local, CPU).

1. Normalise questions; repeated questions form one group.
2. Per group, pick the medoid answer (highest mean ROUGE-L F1 against the rest of the group).
3. Split 90/10 by GROUP (seed fixed) -> no question text appears in both train and test.
   A 5% dev set (of the 90%) is carved out for tuning. IDs saved to data/processed/split.json.
4. Parse source citations per row; normalise the unseen set and flag its overlap with train.
5. Leave-one-out estimate of how well "return the stored answer" does for repeated questions
   (the only honest way to size the exact-match router, since the grouped split has no overlap).

Run:  C:\\nlp_data\\venv\\Scripts\\python.exe src\\phase0_prepare.py
"""
import csv
import json
import os
import random
import re
import statistics as st
import sys
import unicodedata
from collections import defaultdict

from rouge_score import rouge_scorer

sys.path.insert(0, os.path.dirname(__file__))
from sources import parse_sources  # noqa: E402

ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
DATA_DIR = os.environ.get("NLP_DATA_DIR", ROOT)  # where the two raw CSVs live
OUT = os.environ.get("NLP_OUT_DIR", os.path.join(ROOT, "data", "processed"))
SEED = 42
TEST_FRAC = 0.10
DEV_FRAC_OF_TRAIN = 0.05
csv.field_size_limit(10 ** 9)


def norm_question(q):
    """NFKC + lowercase + keep letters/digits/combining marks (so Indic scripts survive) + collapse spaces."""
    q = unicodedata.normalize("NFKC", q or "").lower()
    q = "".join(c if unicodedata.category(c)[0] in "LNM" else " " for c in q)
    return re.sub(r"\s+", " ", q).strip()


def read(name):
    return list(csv.DictReader(open(os.path.join(DATA_DIR, name), encoding="utf-8")))


def main():
    os.makedirs(OUT, exist_ok=True)
    train = read("training_set_15000.csv")
    test = read("unseen_set_1000.csv")
    for r in train:
        r["norm_q"] = norm_question(r["question"])
    for r in test:
        r["norm_q"] = norm_question(r["question"])

    # ---- 1. groups
    gid_of, members = {}, defaultdict(list)
    for i, r in enumerate(train):
        g = gid_of.setdefault(r["norm_q"], "g%05d" % len(gid_of))
        r["group_id"] = g
        members[g].append(i)
    print("rows", len(train), "groups", len(members), "repeated groups", sum(len(v) > 1 for v in members.values()))

    # ---- 2. medoids + pairwise ROUGE-L (also feeds the leave-one-out estimate)
    sc = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
    medoid_idx, loo_scores, loo_random = {}, [], []
    for g, idx in members.items():
        if len(idx) == 1:
            medoid_idx[g] = idx[0]
            continue
        n = len(idx)
        M = [[0.0] * n for _ in range(n)]
        for a in range(n):
            for b in range(a + 1, n):
                f = sc.score(train[idx[a]]["final_answer"], train[idx[b]]["final_answer"])["rougeL"].fmeasure
                M[a][b] = M[b][a] = f
        mean = [sum(M[a]) / (n - 1) for a in range(n)]
        # tie-break (e.g. size-2 groups are symmetric): lowest final_answer_id, deterministic
        best = max(range(n), key=lambda a: (round(mean[a], 9), -int(train[idx[a]]["final_answer_id"], 16)))
        medoid_idx[g] = idx[best]
        # leave-one-out: hold member h out, predict medoid of the remaining members, score vs h
        for h in range(n):
            rest = [a for a in range(n) if a != h]
            if len(rest) == 1:
                pred = rest[0]
            else:
                pred = max(rest, key=lambda a: (round(sum(M[a][b] for b in rest if b != a) / (len(rest) - 1), 9),
                                                -int(train[idx[a]]["final_answer_id"], 16)))
            loo_scores.append(M[h][pred])
            loo_random.append(st.mean(M[h][a] for a in rest))
    for g, i in medoid_idx.items():
        train[i]["is_medoid"] = 1
    for r in train:
        r.setdefault("is_medoid", 0)

    # ---- 3. grouped split
    rng = random.Random(SEED)
    groups = sorted(members)
    rng.shuffle(groups)
    n_rows = len(train)
    test_target = round(TEST_FRAC * n_rows)
    split_of, counts = {}, defaultdict(int)
    for g in groups:
        if counts["test"] < test_target:
            s = "test"
        else:
            s = "train_pool"
        split_of[g] = s
        counts[s] += len(members[g])
    dev_target = round(DEV_FRAC_OF_TRAIN * counts["train_pool"])
    counts["dev"] = 0
    for g in groups:
        if split_of[g] == "train_pool":
            if counts["dev"] < dev_target:
                split_of[g] = "dev"
                counts["dev"] += len(members[g])
            else:
                split_of[g] = "train"
    for r in train:
        r["split"] = split_of[r["group_id"]]
    sizes = {s: sum(1 for r in train if r["split"] == s) for s in ("train", "dev", "test")}
    print("split rows", sizes)
    assert not ({r["norm_q"] for r in train if r["split"] == "test"} &
                {r["norm_q"] for r in train if r["split"] != "test"}), "question leaked across split"

    # ---- 4. citations, unseen set overlap
    for r in train:
        r["n_words_answer"] = len(r["final_answer"].split())
        r["group_size"] = len(members[r["group_id"]])
        cits = [c for c in parse_sources(r["final_answer_sources"]) if c["kind"] in ("zoho", "external")]
        r["citations_json"] = json.dumps(
            [{"doc_id": c["file_id"] or c["url"], "kind": c["kind"], "pages": c["pages"], "name": c["name"],
              "source_type": c["source_type"]} for c in cits], ensure_ascii=False)
    cols = ["question_id", "question", "norm_q", "group_id", "group_size", "is_medoid", "split",
            "final_answer_id", "final_answer", "n_words_answer", "citations_json"]
    with open(os.path.join(OUT, "train_processed.csv"), "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=cols, extrasaction="ignore")
        w.writeheader()
        w.writerows(train)

    in_train = 0
    for r in test:
        g = gid_of.get(r["norm_q"])
        r["in_train"] = int(g is not None)
        r["matched_group_id"] = g or ""
        r["matched_group_size"] = len(members[g]) if g else 0
        in_train += r["in_train"]
    with open(os.path.join(OUT, "test_processed.csv"), "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["question_id", "question", "norm_q", "in_train", "matched_group_id",
                                          "matched_group_size"])
        w.writeheader()
        w.writerows(test)

    json.dump({
        "seed": SEED, "test_frac": TEST_FRAC, "dev_frac_of_train": DEV_FRAC_OF_TRAIN, "rows": sizes,
        "ids": {s: [r["question_id"] for r in train if r["split"] == s] for s in ("train", "dev", "test")},
    }, open(os.path.join(OUT, "split.json"), "w"))

    # ---- 5. report
    wl = [r["n_words_answer"] for r in train]
    rep = {
        "train_rows": n_rows, "groups": len(members),
        "repeated_groups": sum(len(v) > 1 for v in members.values()),
        "split_rows": sizes,
        "unseen_rows": len(test), "unseen_exact_in_train": in_train,
        "answer_words_median": st.median(wl),
        "loo_medoid_rougeL_mean": round(st.mean(loo_scores), 4),
        "loo_random_other_rougeL_mean": round(st.mean(loo_random), 4),
        "loo_pairs": len(loo_scores),
    }
    json.dump(rep, open(os.path.join(OUT, "phase0_report.json"), "w"), indent=2)
    print(json.dumps(rep, indent=2))


if __name__ == "__main__":
    main()

Writing phase0_prepare.py


In [4]:
%%writefile eval_harness.py
"""Single evaluation harness used by every system (baselines, ablations, final model).

    from eval_harness import evaluate
    scores = evaluate(preds, refs)            # dicts {question_id: text}
    scores = evaluate(preds, refs, bertscore=True, meteor=True)   # slower extras

Metrics: ROUGE-1/2/L F1, corpus BLEU (sacrebleu), length ratio, agronomic fact recall/precision/F1
(number+unit pairs, formulation-tagged chemicals such as "dinotefuran 20 SG", binomials), and optionally
METEOR (nltk) and BERTScore-F1. Extras are lazy-imported so the core runs with rouge-score + sacrebleu only.
"""
import re
import statistics as st

from rouge_score import rouge_scorer

_UNIT = {"gm": "g", "gram": "g", "grams": "g", "gms": "g", "litre": "l", "litres": "l", "lit": "l", "liter": "l",
         "liters": "l", "hectare": "ha", "hectares": "ha", "per cent": "%", "percent": "%", "quintals": "q",
         "quintal": "q", "tonnes": "t", "tonne": "t", "day": "days", "kgs": "kg"}
NUM_UNIT_RE = re.compile(
    r"(\d+(?:\.\d+)?)\s*(?:-|–|to)?\s*(\d+(?:\.\d+)?)?\s*"
    r"(kg|kgs|gms?|grams?|g|mg|ml|litres?|liters?|lit|l|quintals?|q|tonnes?|t|ha|hectares?|acre|cm|mm|"
    r"days?|per\s*cent|percent|%|ppm)\b"
    r"(?:\s*(?:/|per)\s*(ha|hectare|acre|litres?|liters?|lit|l|plant|kg|m2|10\s*l))?", re.I)
CHEM_RE = re.compile(r"\b([A-Za-z][A-Za-z\-]{3,})\s+\d+(?:\.\d+)?\s*%?\s*"
                     r"(SG|SL|EC|WP|WG|SC|FS|DP|GR|ZC|OD|EW|CS|ME|WS|DF)\b")
BINOM_RE = re.compile(r"\(([A-Z][a-z]{3,}\s+[a-z]{4,})\)")


def _u(x):
    x = re.sub(r"\s+", " ", x.lower().strip())
    return _UNIT.get(x, x)


def extract_facts(text):
    facts = set()
    for m in NUM_UNIT_RE.finditer(text):
        a, b, unit, per = m.groups()
        facts.add("%s%s%s%s" % (a, "-" + b if b else "", _u(unit), "/" + _u(per) if per else ""))
    for m in CHEM_RE.finditer(text):
        facts.add("chem:" + m.group(1).lower())
    for m in BINOM_RE.finditer(text):
        facts.add("sp:" + m.group(1).lower())
    return facts


def _fact_scores(preds, refs):
    r_list, p_list, f_list = [], [], []
    for p, r in zip(preds, refs):
        fr, fp = extract_facts(r), extract_facts(p)
        if not fr:
            continue
        inter = len(fr & fp)
        rec = inter / len(fr)
        prec = inter / len(fp) if fp else 0.0
        r_list.append(rec)
        p_list.append(prec)
        f_list.append(0.0 if rec + prec == 0 else 2 * rec * prec / (rec + prec))
    n = max(1, len(r_list))
    return {"fact_recall": sum(r_list) / n, "fact_precision": sum(p_list) / n, "fact_f1": sum(f_list) / n,
            "fact_n_refs": len(r_list)}


def evaluate(preds, refs, bertscore=False, meteor=False, bert_model="roberta-large"):
    ids = [i for i in refs if i in preds]
    P = [preds[i] or "" for i in ids]
    R = [refs[i] for i in ids]
    sc = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    agg = {"rouge1": 0.0, "rouge2": 0.0, "rougeL": 0.0}
    for p, r in zip(P, R):
        s = sc.score(r, p)
        for k in agg:
            agg[k] += s[k].fmeasure
    n = max(1, len(ids))
    out = {"n": len(ids), "n_missing": len(refs) - len(ids)}
    out.update({k: v / n for k, v in agg.items()})
    import sacrebleu
    out["bleu"] = sacrebleu.corpus_bleu(P, [R]).score / 100.0
    pl, rl = [len(p.split()) for p in P], [len(r.split()) for r in R]
    out["len_ratio_mean"] = (sum(pl) / n) / max(1e-9, sum(rl) / n)
    out["len_ratio_median_per_item"] = st.median([a / max(1, b) for a, b in zip(pl, rl)]) if ids else 0.0
    out.update(_fact_scores(P, R))
    if meteor:
        from nltk.translate.meteor_score import meteor_score
        out["meteor"] = sum(meteor_score([r.split()], p.split()) for p, r in zip(P, R)) / n
    if bertscore:
        from bert_score import score as bs
        _, _, f1 = bs(P, R, model_type=bert_model, lang="en", batch_size=16, verbose=False)
        out["bertscore_f1"] = float(f1.mean())
    return out


def evaluate_by_group(preds, refs, groups, **kw):
    """groups: {name: set(question_ids)} -> {name: scores}; e.g. novel vs repeated, or per crop."""
    return {g: evaluate(preds, {i: refs[i] for i in ids if i in refs}, **kw) for g, ids in groups.items()}


if __name__ == "__main__":
    t = "Spray dinotefuran 20 SG @ 60 g/acre; seed rate 25 kg/ha. Pest: Amrasca devastans (Amrasca devastans)."
    print(extract_facts(t))
    print(evaluate({"a": t}, {"a": t}))

Writing eval_harness.py


In [5]:
import glob, os
hit = glob.glob("/kaggle/input/**/training_set_15000.csv", recursive=True)
assert hit, "Attach the dataset containing training_set_15000.csv and unseen_set_1000.csv"
os.environ["NLP_DATA_DIR"] = os.path.dirname(hit[0])
os.environ["NLP_OUT_DIR"] = "/kaggle/working/processed"
print("data dir:", os.environ["NLP_DATA_DIR"])

data dir: /kaggle/input/datasets/arth99/nlp-qna-raw


## Run Phase 0 (grouping, medoids, grouped split, citation parsing; ~1-2 min)

In [6]:
import phase0_prepare
phase0_prepare.main()

rows 15000 groups 14389 repeated groups 501
split rows {'train': 12825, 'dev': 675, 'test': 1500}
{
  "train_rows": 15000,
  "groups": 14389,
  "repeated_groups": 501,
  "split_rows": {
    "train": 12825,
    "dev": 675,
    "test": 1500
  },
  "unseen_rows": 1000,
  "unseen_exact_in_train": 438,
  "answer_words_median": 412.0,
  "loo_medoid_rougeL_mean": 0.6069,
  "loo_random_other_rougeL_mean": 0.6054,
  "loo_pairs": 1112
}


## Sanity checks and evaluation-harness self-test

In [7]:
import csv, json
csv.field_size_limit(10**9)
P = "/kaggle/working/processed/"
rows = list(csv.DictReader(open(P + "train_processed.csv", encoding="utf-8")))
sp = {s: {r["norm_q"] for r in rows if r["split"] == s} for s in ("train", "dev", "test")}
assert not (sp["test"] & (sp["train"] | sp["dev"])), "leak"
assert not (sp["dev"] & sp["train"]), "leak"
print({s: sum(r["split"] == s for r in rows) for s in sp}, "- no question leaks across splits")
print("medoids per group ok:", len({r["group_id"] for r in rows if r["is_medoid"] == "1"}) == len({r["group_id"] for r in rows}))
from eval_harness import evaluate
test = [r for r in rows if r["split"] == "test"]
refs = {r["question_id"]: r["final_answer"] for r in test}
print("self-score (should be 1.0):", {k: round(v, 3) for k, v in evaluate(refs, refs).items() if k in ("rouge1", "rougeL", "bleu", "fact_f1")})
print(json.dumps(json.load(open(P + "phase0_report.json")), indent=1))

{'train': 12825, 'dev': 675, 'test': 1500} - no question leaks across splits
medoids per group ok: True
self-score (should be 1.0): {'rouge1': 1.0, 'rougeL': 1.0, 'bleu': 1.0, 'fact_f1': 1.0}
{
 "train_rows": 15000,
 "groups": 14389,
 "repeated_groups": 501,
 "split_rows": {
  "train": 12825,
  "dev": 675,
  "test": 1500
 },
 "unseen_rows": 1000,
 "unseen_exact_in_train": 438,
 "answer_words_median": 412.0,
 "loo_medoid_rougeL_mean": 0.6069,
 "loo_random_other_rougeL_mean": 0.6054,
 "loo_pairs": 1112
}


## Save outputs
*Save Version -> Save & Run All*, then publish `processed/` as a Kaggle Dataset (e.g. `nlp-qna-processed`). Later notebooks
attach it for the train/dev/test splits. Also copy `eval_harness.py` from the Output tab - every later notebook reuses it.